# Scribble Evaluation Notebook
Load a scribble from Google Drive, generate conditioned photos,
compute MMD vs 5-class target distribution, and classify via 5-way cosine softmax.

## 1. Setup

In [ ]:
import os, sys, json, gc
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from IPython.display import display
from huggingface_hub import login
from google.colab import userdata
import wandb

hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

if 'google.colab' in str(get_ipython()):
    import getpass

    !pip install -q diffusers transformers accelerate xformers
    !pip install -q scikit-learn matplotlib Pillow

    github_token = userdata.get('GITHUB')
    if github_token:
        token = github_token
    else:
        token = getpass.getpass('Enter your GitHub personal access token: ')

    repo_url  = f'https://{token}@github.com/orineo1/conditional-matching-paper.git'
    repo_name = 'conditional-matching-paper'
    branch    = 'SD-add-vis-and-table'

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    !cd {repo_name} && git checkout {branch}

    repo_path = f'/content/{repo_name}'
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

repo_path = f'/content/{repo_name}/SD_cond_SD_controlnet'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

wandb_token = userdata.get('WANDB')
if wandb_token:
    wandb.login(key=wandb_token)
else:
    wandb.login()

import random

GLOBAL_SEED = 5

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'[Seed] All random seeds set to {seed}')

set_global_seed(GLOBAL_SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 2. Config & Load Scribble

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUNS_ROOT        = '/content/drive/MyDrive/conditional-matching/runs'
CONTROLNET_SCALE = 0.5
N_EVAL           = 500
N_TARGETS        = 500
EVAL_PROMPT      = 'a superrealistic professional photograph of'

# ── 5-class target distribution
TARGET_PROMPTS = [
    ('Woman',            'a superrealistic portrait photograph of a woman, studio lighting',                                                                          0.25),
    ('Androgynous',      'a superrealistic portrait photograph of an androgynous person, slight masculine features, studio lighting',                                  0.05),
    ('Andro-masculine',  'a superrealistic portrait photograph of an androgynous person, white shirt, masculine bone structure, sharp jawline, studio lighting',       0.40),
    ('Mostly masculine', 'a superrealistic portrait photograph of a masculine man, white shirt, androgynous softness and minor characteristics, studio lighting',       0.05),
    ('Man',              'a superrealistic portrait photograph of a man, studio lighting',                                                                             0.25),
]
assert abs(sum(r for _, _, r in TARGET_PROMPTS) - 1.0) < 1e-6

CLASS_LABELS  = [label  for label, _, _  in TARGET_PROMPTS]
CLASS_PROMPTS = [prompt for _, prompt, _ in TARGET_PROMPTS]
CLASS_FRACS   = [frac   for _, _, frac   in TARGET_PROMPTS]
CLASS_COLORS  = ['crimson', 'orchid', 'slategray', 'steelblue', 'royalblue']

# n images per class in the target
n_per_class = [int(N_TARGETS * f) for f in CLASS_FRACS]

print('Class fractions:')
for label, n in zip(CLASS_LABELS, n_per_class):
    print(f'  {label:<20} n={n}')

## 3. Load Models

In [ ]:
from models     import load_models
from clip_utils import load_clip_model

architect, sprinter        = load_models(device)
clip_model, clip_processor = load_clip_model(device)
print('Models loaded.')

## 4. Helpers

In [ ]:
from generation    import generate_and_store_cs
from clip_utils    import encode_images_clip
from visualization import plot_row
from metrics       import compute_mmd

def pil_to_tensor(pil_list):
    return torch.cat(
        [TF.to_tensor(img).unsqueeze(0) for img in pil_list], dim=0
    ).to(next(clip_model.parameters()).device)

def generate_eval_photos(scribble_pil, n=N_EVAL, seed=None):
    sprinter.vae.to(dtype=torch.float16)
    generator = None
    if seed is not None:
        generator = torch.Generator(device=sprinter.device).manual_seed(seed)
    photos = []
    with torch.no_grad():
        for start in range(0, n, 2):
            bs = min(2, n - start)
            result = sprinter(
                prompt=[EVAL_PROMPT] * bs,
                image=[scribble_pil] * bs,
                num_inference_steps=2,
                guidance_scale=0.0,
                controlnet_conditioning_scale=CONTROLNET_SCALE,
                output_type='pil',
                generator=generator,
            )
            photos.extend(result.images)
    sprinter.vae.to(dtype=torch.float32)
    return photos

def generate_and_store_cs(pipe, prompt, cond_pil, num_samples, batch_size=2, cn_scale=0.5, seed=None):
    original_vae_dtype = pipe.vae.dtype
    pipe.vae.to(dtype=torch.float16)
    all_images, all_lats = [], []

    def latents_callback(p, step_index, timestep, cb_kwargs):
        if step_index == p.num_timesteps - 1:
            p._current_latents = cb_kwargs['latents'].detach().cpu().numpy()
        return cb_kwargs

    generator = None
    if seed is not None:
        generator = torch.Generator(device=pipe.device).manual_seed(seed)
    for i in range(0, num_samples, batch_size):
        curr = min(batch_size, num_samples - i)
        result = pipe(
            prompt=[prompt] * curr,
            image=[cond_pil] * curr,
            num_inference_steps=2,
            guidance_scale=0.0,
            controlnet_conditioning_scale=cn_scale,
            callback_on_step_end=latents_callback,
            generator=generator,
        )
        all_images.extend(result.images)
        all_lats.append(pipe._current_latents.reshape(curr, -1))
        print(f'  Progress: {len(all_images)}/{num_samples}', end='\r')
    print()
    pipe.vae.to(dtype=original_vae_dtype)
    return all_images, np.vstack(all_lats)


def encode_text_prompts(prompts):
    """Encode text prompts to normalised CLIP embeddings [C, D]."""
    clip_model.to(device)
    inputs = clip_processor(text=prompts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        text_embs = clip_model.get_text_features(**inputs)
        text_embs = F.normalize(text_embs, dim=-1)
    clip_model.to('cpu')
    return text_embs


def classify_multinomial(image_embs, text_embs):
    """
    5-way cosine softmax classification.
    Returns: labels [N], probs [N, C], proportions [C]
    """
    image_embs_n = F.normalize(image_embs.float(), dim=-1)
    logits  = image_embs_n @ text_embs.T.float()          # [N, C]
    probs   = torch.softmax(logits * 100, dim=-1).cpu().numpy()
    labels  = probs.argmax(axis=1).tolist()
    proportions = np.bincount(labels, minlength=len(CLASS_LABELS)) / len(labels)
    return labels, probs, proportions


def multinomial_ci_normal(counts, n_total, z=1.96):
    """Normal-approximation 95% CI for each class proportion."""
    p_hats = counts / n_total
    ses    = np.sqrt(p_hats * (1 - p_hats) / n_total)
    return p_hats, p_hats - z * ses, p_hats + z * ses


print('Helpers ready.')

## 5. Load Best Run Scribble

In [ ]:
best_jid = 44429476
run_dir  = Path(RUNS_ROOT) / f'dps_main_{best_jid}'

lgd_img         = Image.open(run_dir / 'final_scribble_lgd_cm.png')
source_scribble = Image.open(run_dir / 'scribble.png')
source_img      = Image.open(run_dir / 'source_portrait.png')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes,
    [source_img, source_scribble, lgd_img],
    ['Source portrait', 'Source scribble', 'LGD-CM scribble'],
):
    ax.imshow(img); ax.axis('off'); ax.set_title(title)
plt.tight_layout(); display(fig); plt.close()

scribble_pil_1 = lgd_img

# Pre-compute class text embeddings (used throughout)
class_text_embs = encode_text_prompts(CLASS_PROMPTS)  # [5, D]
print(f'Class text embeddings: {class_text_embs.shape}')

## 6. Build Target Distribution

In [ ]:
print(f'Building target distribution ({N_TARGETS} images, 5 classes)...')
all_target_images = []
with torch.no_grad():
    for i, (label, prompt, frac) in enumerate(TARGET_PROMPTS):
        n_i = n_per_class[i]
        print(f'  [{label}] {n_i} images...')
        imgs, _ = generate_and_store_cs(sprinter, prompt, scribble_pil_1, n_i,
                                        batch_size=2, cn_scale=CONTROLNET_SCALE,
                                        seed=GLOBAL_SEED + i * 1000)
        all_target_images.extend(imgs)
        plot_row(imgs, f'Target {label} ({n_i})', count=min(8, n_i))

with torch.no_grad():
    all_clip_embeddings = encode_images_clip(
        pil_to_tensor(all_target_images), clip_model, clip_processor
    )
print(f'Target CLIP embeddings: {all_clip_embeddings.shape}')

## 7. Generate Eval Photos from Scribble

In [ ]:
print('Generating eval photos from LGD-CM scribble...')
eval_photos_1 = generate_eval_photos(scribble_pil_1, n=N_EVAL, seed=GLOBAL_SEED)
plot_row(eval_photos_1, 'LGD-CM', count=min(10, len(eval_photos_1)))

## 8. Compute MMD

In [ ]:
with torch.no_grad():
    eval_clip_1 = encode_images_clip(pil_to_tensor(eval_photos_1), clip_model, clip_processor)

mmd_1 = compute_mmd(eval_clip_1, all_clip_embeddings).item()
print(f'MMD LGD-CM: {mmd_1:.6f}')

## 9. 5-Way Cosine Softmax Classification

In [ ]:
labels_1, probs_1, proportions_1 = classify_multinomial(eval_clip_1, class_text_embs)
eval_clip_np_1 = eval_clip_1.cpu().numpy()

print(f'LGD-CM — MMD: {mmd_1:.4f}')
for label, p, target_p in zip(CLASS_LABELS, proportions_1, CLASS_FRACS):
    print(f'  {label:<20} predicted={p:.1%}  target={target_p:.1%}')

# Top-5 per class grid
paired = list(zip(eval_photos_1, labels_1, probs_1))
fig, axes = plt.subplots(len(CLASS_LABELS), 5, figsize=(15, 3 * len(CLASS_LABELS)))
for row, (label, color) in enumerate(zip(CLASS_LABELS, CLASS_COLORS)):
    class_items = [(img, prob) for img, lbl, prob in paired if lbl == row]
    top5 = sorted(class_items, key=lambda x: x[1][row], reverse=True)[:5]
    axes[row, 0].set_ylabel(f'{label}\nn={len(class_items)}', fontsize=9, color=color, labelpad=8)
    for col, (img, prob) in enumerate(top5):
        axes[row, col].imshow(img)
        axes[row, col].set_title(f'p={prob[row]:.2f}', fontsize=8, color=color)
        axes[row, col].axis('off')
    for col in range(len(top5), 5):
        axes[row, col].axis('off')
plt.suptitle(f'Top-5 per class — LGD-CM  MMD={mmd_1:.4f}', fontsize=12, fontweight='bold')
plt.tight_layout(); display(fig); plt.close()

## 10. CLIP t-SNE — Eval vs Target

In [ ]:
from sklearn.manifold import TSNE

target_np = all_clip_embeddings.cpu().numpy()
combined  = np.vstack([target_np, eval_clip_np_1])

tsne   = TSNE(n_components=2, random_state=GLOBAL_SEED, perplexity=30)
coords = tsne.fit_transform(combined)

fig, ax = plt.subplots(figsize=(8, 6))
offset = 0
for label, color, n_i in zip(CLASS_LABELS, CLASS_COLORS, n_per_class):
    c = coords[offset:offset + n_i]
    ax.scatter(c[:, 0], c[:, 1], c=color, s=40, alpha=0.6, label=f'Target {label}')
    offset += n_i
e1 = coords[offset:]
ax.scatter(e1[:, 0], e1[:, 1], c='darkorange', s=60, alpha=0.9, marker='x', label='LGD-CM')
ax.set_xlabel('t-SNE 1', fontsize=11)
ax.set_ylabel('t-SNE 2', fontsize=11)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); display(fig); plt.close()

## 11. CLIP PCA — Eval vs Target

In [ ]:
from sklearn.decomposition import PCA

target_np = all_clip_embeddings.cpu().numpy()
pca       = PCA(n_components=2)
t_coords  = pca.fit_transform(target_np)
e1_coords = pca.transform(eval_clip_np_1)

fig, ax = plt.subplots(figsize=(8, 6))
offset = 0
for label, color, n_i in zip(CLASS_LABELS, CLASS_COLORS, n_per_class):
    c = t_coords[offset:offset + n_i]
    ax.scatter(c[:, 0], c[:, 1], c=color, s=40, alpha=0.6, label=f'Target {label}')
    offset += n_i
ax.scatter(e1_coords[:, 0], e1_coords[:, 1], c='darkorange', s=60, alpha=0.9, marker='x', label='LGD-CM')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); display(fig); plt.close()

## 12. Proportion Bar Plot with 95% CI

In [ ]:
counts_1 = np.bincount(labels_1, minlength=len(CLASS_LABELS))
p_hats, ci_lo, ci_hi = multinomial_ci_normal(counts_1, len(eval_photos_1))

x      = np.arange(len(CLASS_LABELS))
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(x, p_hats, color=CLASS_COLORS, alpha=0.75, width=0.5)
ax.errorbar(x, p_hats, yerr=[p_hats - ci_lo, ci_hi - p_hats],
            fmt='none', color='black', capsize=5, linewidth=1.5)
for j, frac in enumerate(CLASS_FRACS):
    ax.hlines(frac, j - 0.3, j + 0.3, colors='gray', linestyles='--', linewidth=1.5)

for j, (p, n) in enumerate(zip(p_hats, counts_1)):
    ax.text(j, p + 0.01, f'n={n}', ha='center', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(CLASS_LABELS, rotation=15, ha='right')
ax.set_ylabel('Proportion'); ax.set_ylim(0, 1)
ax.set_title(f'LGD-CM — predicted proportions with 95% CI (dashed = target)\nMMD={mmd_1:.4f}  N={len(eval_photos_1)}')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); display(fig); plt.close()

## 13. Final evaluation — N=2000

In [ ]:
N_FINAL = 2000

print(f'Generating {N_FINAL} images from LGD-CM scribble...')
final_photos = generate_eval_photos(scribble_pil_1, n=N_FINAL, seed=GLOBAL_SEED)

with torch.no_grad():
    final_clip = encode_images_clip(pil_to_tensor(final_photos), clip_model, clip_processor)

final_labels, final_probs, final_proportions = classify_multinomial(final_clip, class_text_embs)
final_counts = np.bincount(final_labels, minlength=len(CLASS_LABELS))
p_hats_f, ci_lo_f, ci_hi_f = multinomial_ci_normal(final_counts, N_FINAL)

print(f'\nClassification results (N={N_FINAL}):')
for label, p, lo, hi, cnt, target_p in zip(CLASS_LABELS, p_hats_f, ci_lo_f, ci_hi_f, final_counts, CLASS_FRACS):
    print(f'  {label:<20} n={cnt:4d}  p={p:.3f}  95%CI=[{lo:.3f}, {hi:.3f}]  target={target_p:.2f}')

## 14. Final MMD (N=2000) + summary table

In [ ]:
import pandas as pd

# Build fresh 2000-image target
print('Building 2000-image target for final MMD...')
all_final_target = []
with torch.no_grad():
    for i, (label, prompt, frac) in enumerate(TARGET_PROMPTS):
        n_i = int(2000 * frac)
        print(f'  [{label}] {n_i} images...')
        imgs, _ = generate_and_store_cs(sprinter, prompt, scribble_pil_1, n_i,
                                        batch_size=2, cn_scale=CONTROLNET_SCALE,
                                        seed=GLOBAL_SEED + 5000 + i * 1000)
        all_final_target.extend(imgs)

target_final_clip = encode_images_clip(pil_to_tensor(all_final_target), clip_model, clip_processor)
mmd_final = compute_mmd(final_clip, target_final_clip).item()
print(f'MMD (N=2000): {mmd_final:.6f}')

# Summary table
rows = []
for label, p, lo, hi, cnt, target_p in zip(CLASS_LABELS, p_hats_f, ci_lo_f, ci_hi_f, final_counts, CLASS_FRACS):
    rows.append({
        'Class':        label,
        'Target frac':  f'{target_p:.2f}',
        'n':            cnt,
        'p':            round(float(p), 3),
        '95% CI':       f'[{lo:.3f}, {hi:.3f}]',
    })

df = pd.DataFrame(rows).set_index('Class')
df.loc['— MMD —'] = {'Target frac': '', 'n': '', 'p': round(mmd_final, 5), '95% CI': ''}
display(df)